# 01 — Download and harmonize YRBSS opioid misuse estimates

This notebook fetches opioid-related prevalence estimates from CDC YRBSS public API endpoints and produces an annual (biennial survey years) cleaned table for comparison to NSDUH.

In [ ]:
import os
from pathlib import Path
import requests
import pandas as pd

pd.set_option("display.max_columns", 120)

In [ ]:
DATA_ROOT = Path('..') / 'data'
RAW_DIR = DATA_ROOT / 'raw' / 'yrbss'
DERIVED_DIR = DATA_ROOT / 'derived' / 'yrbss'
RAW_DIR.mkdir(parents=True, exist_ok=True)
DERIVED_DIR.mkdir(parents=True, exist_ok=True)

RAW_DIR, DERIVED_DIR

## Pull YRBSS from CDC data API

The endpoint ID can change over time. We expose it as a parameter so the notebook is resilient if CDC republishes the dataset.

In [ ]:
# National YRBSS indicator dataset on data.cdc.gov (update if CDC rotates view IDs)
YRBSS_VIEW_ID = os.environ.get('YRBSS_VIEW_ID', '7xiw-8vpk')
BASE = f'https://data.cdc.gov/resource/{YRBSS_VIEW_ID}.csv'

params = {
    '$limit': 500000,
    '$order': 'year',
}

raw_path = RAW_DIR / f'yrbss_{YRBSS_VIEW_ID}.csv'
try:
    resp = requests.get(BASE, params=params, timeout=120)
    resp.raise_for_status()
    raw_path.write_bytes(resp.content)
    print(f'Saved: {raw_path} ({raw_path.stat().st_size/1e6:.1f} MB)')
except Exception as e:
    print(f'API download failed: {e}')
    if raw_path.exists():
        print(f'Using existing file: {raw_path}')
    else:
        fallback = sorted(RAW_DIR.glob('yrbss_*.csv'))
        if fallback:
            raw_path = fallback[-1]
            print(f'Using fallback local file: {raw_path}')
        else:
            raise RuntimeError('No YRBSS file available locally; retry when network/API access is available.')

In [ ]:
df = pd.read_csv(raw_path)
print(df.shape)
df.head(3)

## Identify opioid misuse indicators

YRBSS label/variable names vary by release; we use flexible text matching to isolate prescription pain medicine misuse / heroin use indicators at the national level.

In [ ]:
question_cols = [c for c in df.columns if 'question' in c.lower() or 'topic' in c.lower() or 'class' in c.lower() or 'indicator' in c.lower()]
value_cols = [c for c in df.columns if c.lower() in {'data_value','value','estimate','percent'} or 'value' in c.lower()]
print('question-like:', question_cols[:8])
print('value-like:', value_cols[:8])

In [ ]:
# Heuristic column mapping
def pick(cols, candidates):
    for cand in candidates:
        for c in cols:
            if c.lower() == cand.lower():
                return c
    for cand in candidates:
        for c in cols:
            if cand.lower() in c.lower():
                return c
    return None

year_col = pick(df.columns, ['year'])
question_col = pick(df.columns, ['question','topic','indicator','short_question_text'])
value_col = pick(df.columns, ['data_value','value','estimate'])
low_col = pick(df.columns, ['low_confidence_limit','low_ci','ci_low'])
high_col = pick(df.columns, ['high_confidence_limit','high_ci','ci_high'])
location_col = pick(df.columns, ['locationdesc','location','site'])
strat_col = pick(df.columns, ['stratification1','stratum','sex'])

print({'year':year_col,'question':question_col,'value':value_col,'low':low_col,'high':high_col,'location':location_col,'strat':strat_col})

In [ ]:
flt = df.copy()
if location_col is not None:
    flt = flt[flt[location_col].astype(str).str.contains('national', case=False, na=False)]
if strat_col is not None:
    flt = flt[flt[strat_col].astype(str).str.contains('overall|total', case=False, na=False)]

q = flt[question_col].astype(str).str.lower() if question_col else pd.Series('', index=flt.index)
opioid_mask = q.str.contains('prescription pain|pain medicine|opioid|heroin', na=False)
flt = flt[opioid_mask].copy()

trend = flt[[year_col, question_col, value_col, low_col, high_col]].copy()
trend.columns = ['year','indicator','prevalence_pct','ci_low','ci_high']
trend['year'] = pd.to_numeric(trend['year'], errors='coerce')
trend['prevalence_pct'] = pd.to_numeric(trend['prevalence_pct'], errors='coerce')
trend['ci_low'] = pd.to_numeric(trend['ci_low'], errors='coerce')
trend['ci_high'] = pd.to_numeric(trend['ci_high'], errors='coerce')
trend = trend.dropna(subset=['year','prevalence_pct']).sort_values(['indicator','year'])
trend.head(10)

In [ ]:
# Keep the primary prescription pain medicine misuse trend if present
primary = trend[trend['indicator'].str.contains('prescription pain|pain medicine', case=False, na=False)].copy()
if primary.empty:
    primary = trend.copy()

primary['source'] = 'YRBSS'
primary.to_csv(DERIVED_DIR / 'yrbss_opioid_trends.csv', index=False)
print('Saved:', DERIVED_DIR / 'yrbss_opioid_trends.csv')
primary.tail()